# RAG (Retrieval-Augmented Generation) - Comprehensive Implementation Guide

This notebook covers:
- **Basic Implementation**: Simple, educational version
- **Advanced Implementation**: Production-ready patterns
- **Real-World Examples**: How companies use this in production
- **Integration**: Using popular libraries

Source: `llm/concepts/rag.md`

## Setup & Imports

In [ ]:
import torch
import torch.nn as nn
import numpy as np

print("Libraries loaded successfully")

## 1. Basic Implementation

Simple, educational version to understand core concepts.

In [ ]:
# Basic RAG Pipeline
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

class BasicRAG:
    """Simple retrieval-augmented generation"""

    def __init__(self):
        self.embedder = SentenceTransformer('all-MiniLM-L6-v2')
        self.documents = []
        self.embeddings = None

    def add_documents(self, docs):
        self.documents = docs
        self.embeddings = self.embedder.encode(docs)

    def retrieve(self, query, top_k=3):
        query_emb = self.embedder.encode([query])[0]
        scores = cosine_similarity([query_emb], self.embeddings)[0]
        top_idx = scores.argsort()[::-1][:top_k]
        return [self.documents[i] for i in top_idx]

    def generate_prompt(self, query):
        context = "\n".join(self.retrieve(query, top_k=3))
        prompt = f"""Context: {context}

Question: {query}

Answer:"""
        return prompt

# Usage
rag = BasicRAG()
docs = [
    "Python is a programming language",
    "Machine learning uses data",
    "Neural networks learn patterns"
]
rag.add_documents(docs)
prompt = rag.generate_prompt("What is ML?")
print(prompt)

## 2. Advanced Implementation

Production-ready patterns with optimization and error handling.

In [ ]:
# Advanced RAG with Re-ranking
from sentence_transformers import CrossEncoder

class AdvancedRAG:
    """RAG with dense + sparse retrieval and re-ranking"""

    def __init__(self):
        from sentence_transformers import SentenceTransformer
        self.dense_embedder = SentenceTransformer('all-MiniLM-L6-v2')
        self.reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
        self.documents = []

    def add_documents(self, docs):
        self.documents = docs

    def dense_retrieve(self, query, top_k=10):
        """Dense retrieval: semantic similarity"""
        query_emb = self.dense_embedder.encode([query])[0]
        doc_embs = self.dense_embedder.encode(self.documents)

        from sklearn.metrics.pairwise import cosine_similarity
        scores = cosine_similarity([query_emb], doc_embs)[0]
        top_idx = scores.argsort()[::-1][:top_k]
        return [self.documents[i] for i in top_idx]

    def rerank(self, query, candidates, top_k=3):
        """Re-rank candidates using cross-encoder"""
        pairs = [[query, doc] for doc in candidates]
        scores = self.reranker.predict(pairs)
        sorted_idx = scores.argsort()[::-1][:top_k]
        return [candidates[i] for i in sorted_idx]

    def retrieve_best(self, query, top_k=3):
        candidates = self.dense_retrieve(query, top_k=10)
        return self.rerank(query, candidates, top_k=top_k)

# Usage
rag = AdvancedRAG()
docs = ["Python code", "Java code", "C++ code", "Learning models"]
rag.add_documents(docs)
best = rag.retrieve_best("programming language")
print(f"Best retrieved: {best}")

## Real-World: Llm

How this is used in production systems.

In [ ]:
# Real-World: RAG with LLM Generation
from transformers import pipeline
from sentence_transformers import SentenceTransformer

class ProductionRAG:
    """Production RAG with LLM for answer generation"""

    def __init__(self):
        self.retriever = SentenceTransformer('all-MiniLM-L6-v2')
        self.llm = pipeline("text-generation", model="gpt2")
        self.documents = []

    def add_documents(self, docs):
        self.documents = docs

    def retrieve_context(self, query, top_k=3):
        query_emb = self.retriever.encode(query)
        doc_embs = self.retriever.encode(self.documents)

        from sklearn.metrics.pairwise import cosine_similarity
        scores = cosine_similarity([query_emb], doc_embs)[0]
        top_idx = scores.argsort()[::-1][:top_k]
        return [self.documents[i] for i in top_idx]

    def answer(self, query):
        context = "\n".join(self.retrieve_context(query))
        prompt = f"Context: {context}\n\nQuestion: {query}\n\nAnswer:"

        response = self.llm(prompt, max_length=100)
        return response[0]['generated_text']

# Usage
rag = ProductionRAG()
rag.add_documents([
    "Paris is the capital of France",
    "Tokyo is the capital of Japan",
    "Berlin is the capital of Germany"
])
answer = rag.answer("What is the capital of France?")
print(f"Answer: {answer}")

## Real-World: Vectordb

How this is used in production systems.

In [ ]:
# Real-World: RAG with Vector Database
from pinecone import Pinecone
from sentence_transformers import SentenceTransformer

class VectorDBRAG:
    """RAG using vector database for scalability"""

    def __init__(self, api_key):
        # Note: requires Pinecone account
        # self.pc = Pinecone(api_key=api_key)
        # self.index = self.pc.Index("llm-index")
        self.embedder = SentenceTransformer('all-MiniLM-L6-v2')

    def index_documents(self, documents):
        """Index documents in vector DB"""
        embeddings = self.embedder.encode(documents)

        # Upsert to Pinecone (requires setup)
        # self.index.upsert([(str(i), emb.tolist()) for i, emb in enumerate(embeddings)])

    def retrieve(self, query, top_k=5):
        """Query vector database"""
        query_emb = self.embedder.encode(query)

        # results = self.index.query(query_emb.tolist(), top_k=top_k)
        # return results

# For production: use Pinecone, Weaviate, or Milvus
# Handles millions of documents with millisecond latency

## Resources & Next Steps

- **Detailed Explanation**: See `llm/concepts/rag.md`
- **Interview Questions**: Review Q&A in markdown file
- **Real-World Examples**: See companies section in markdown
- **Experiment**: Modify the code above and run cells

### Concepts to explore next:
- Related concepts (see markdown file)
- Cross-concept combinations
- Integration with other techniques